

## โจทย์การบ้าน: การพัฒนา Two-Agent System

### วัตถุประสงค์

ให้นักศึกษาออกแบบและสร้างระบบ Agent จำนวน 2 ตัวเพื่อทำงานร่วมกัน โดยมีเป้าหมายเพื่อตอบคำถามที่ซับซ้อนได้อย่างถูกต้องและมีคุณภาพสูง

1.  **Agent 1 (ReAct Agent)**: มีหน้าที่หลักในการรวบรวมข้อมูลและคำนวณ โดยใช้เทคนิค ReAct (Reasoning and Acting) ในการตัดสินใจว่าจะใช้เครื่องมือ (Tools) ใดเพื่อหาข้อมูลที่จำเป็น Agent 1 ควรจะสามารถ:

      * **ค้นหาข้อมูล**: ดึงข้อมูลจากฐานความรู้ (เช่น Wikipedia) หรือเว็บเพจที่กำหนด
      * **คำนวณ**: ใช้เครื่องมือคำนวณเพื่อแก้ปัญหาทางคณิตศาสตร์
      * **สังเคราะห์คำตอบ**: รวบรวมข้อมูลดิบที่ได้จากเครื่องมือและสร้างคำตอบเบื้องต้น

2.  **Agent 2 (Self-Reflecting Agent)**: มีหน้าที่ประเมินและปรับปรุงคำตอบที่ได้จาก Agent 1 Agent 2 ควรจะสามารถ:

      * **วิเคราะห์คำตอบ**: ตรวจสอบคำตอบเบื้องต้นที่สร้างโดย Agent 1 เพื่อหาข้อผิดพลาด ความไม่สมบูรณ์ หรือจุดที่สามารถปรับปรุงได้
      * **สร้าง Feedback**: สร้าง feedback ที่เฉพาะเจาะจงเพื่อชี้แนะแนวทางในการแก้ไขคำตอบ
      * **ปรับปรุงคำตอบ**: ใช้ feedback ที่สร้างขึ้นเพื่อปรับปรุงคำตอบให้ถูกต้องและสมบูรณ์ยิ่งขึ้น

### คำถามสำหรับทดสอบ (จำนวน 10 ข้อ)

ให้นำ Agent ทั้งสองมาทดสอบด้วยชุดคำถามต่อไปนี้ โดยคำถามเหล่านี้ถูกออกแบบมาให้ต้องใช้ทั้งการค้นหาข้อมูล การคำนวณ และการสังเคราะห์ข้อมูลที่ซับซ้อน

```python
questions = [
    "ผลคูณของ 147 กับ 258 คือเท่าไหร่?",
    "สรุปประวัติของ AI จากหน้า Wikipedia เป็นภาษาไทย",
    "Elon Musk ก่อตั้งบริษัทใดบ้าง และปัจจุบันมีตำแหน่งอะไรในแต่ละบริษัท?",
    "ค้นหาและสรุปข้อมูลล่าสุดเกี่ยวกับ iPhone รุ่นใหม่",
    "คำนวณค่าของ (15 + 7) * 3 / 2",
    "ข้อมูลเศรษฐกิจของประเทศไทย ณ ปี 2024 เป็นอย่างไรบ้าง? (ให้อ้างอิงแหล่งที่มา)",
    "ดาวเคราะห์ดวงใดในระบบสุริยะที่มีขนาดใหญ่ที่สุดและมีจำนวนดวงจันทร์เท่าใด?",
    "สี่เหลี่ยมจัตุรัสที่มีพื้นที่ 256 ตารางเมตร จะมีความยาวเส้นรอบรูปเท่าใด?",
    "อธิบายแนวคิดหลักของ 'การเรียนรู้ของเครื่อง' (Machine Learning) พร้อมยกตัวอย่าง 2 ตัวอย่าง",
    "เปรียบเทียบข้อดีและข้อเสียของภาษา Python และ Java"
]
```


In [1]:
import IPython
import sys

def clean_notebook():
    IPython.display.clear_output(wait=True)
    print("Notebook cleaned.")

!pip install openai wikipedia sympy requests -q

# Clean up the notebook
clean_notebook()


Notebook cleaned.


In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

True

In [3]:
import os
from openai import OpenAI
import wikipedia
from sympy import sympify
import requests
from datetime import date
import random

# Set up OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
model_name = "gpt-4o"  # Using 'gpt-4o' as a stand-in; change if needed to 'gpt-4.1' or another model

In [4]:
import wikipedia
from sympy import sympify
import requests

# ฟังก์ชันคำนวณค่าทางคณิตศาสตร์
tools = {
    "calculator": lambda expr: str(sympify(expr)),
    
    # ฟังก์ชันดึงข้อมูลจาก Wikipedia โดยใช้ summary
    "wikipedia_lookup": lambda query: str(wikipedia.search(query)),

    # ฟังก์ชันดึงข้อมูลสรุปจาก Wikipedia หน้าแรกที่พบ
    "get_wikipedia_page": lambda query: {
        # หา title ที่ใกล้เคียงจากการค้นหา
        'search_results': wikipedia.search(query),
        'summary': get_wikipedia_summary(query)
    },
    
    # ฟังก์ชันรวบรวมข้อมูลและคำนวณจากเครื่องมือ
    "data_gathering_and_calculating": lambda query, expr=None: {
        "search_results": tools["get_wikipedia_page"](query)['search_results'],
        "calculation_result": tools["calculator"](expr) if expr else "No expression provided"
    }
}

# ฟังก์ชันที่ใช้ในการดึงข้อมูลสรุปจาก Wikipedia
def get_wikipedia_summary(query):
    try:
        # ค้นหาเพจจาก Wikipedia
        search_results = wikipedia.search(query)
        if not search_results:
            return "No page found for query."
        
        # เอาผลลัพธ์แรกที่เจอ
        page_title = search_results[0]
        summary = wikipedia.summary(page_title, sentences=5)  # จำกัดคำสรุปที่ 5 ประโยค
        return summary
    except Exception as e:
        return f"Error fetching Wikipedia page: {e}"

# ทดสอบฟังก์ชัน get_wikipedia_page
print(tools["get_wikipedia_page"]("Artificial intelligence"))


{'search_results': ['Artificial intelligence', 'Artificial general intelligence', 'Generative artificial intelligence', 'A.I. Artificial Intelligence', 'History of artificial intelligence', 'Distributed artificial intelligence', 'Existential risk from artificial intelligence', 'Applications of artificial intelligence', 'Timeline of artificial intelligence', 'Philosophy of artificial intelligence'], 'summary': 'Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.\nHigh-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used b

In [5]:
system_prompt = """
You are a ReAct agent that answers questions by reasoning step by step and using tools when needed.

Available tools:
- calculator: Useful for math calculations. Input: a mathematical expression like "(15 + 7) * 3 / 2".
- browse_page: Fetch content from a webpage. Input: a URL.
- wikipedia_lookup: Search Wikipedia for page titles. Input: a query to search. Example: wikipedia_lookup "History of artificial intelligence"
- get_wikipedia_page: Get content from a Wikipedia page. Input: the exact page title. Example: get_wikipedia_page "History of artificial intelligence"
- data_gathering_and_calculating: Combine searching for information and performing calculations. Input: a query for information and an optional mathematical expression.

Always use this format:
Question: [the question]
Thought: [your reasoning]
Action: [tool_name] [input]
(Stop after Action; you'll get an Observation next.)

After getting observations, continue with a new Thought.

When you have enough information for the final answer, output:
Final Answer: [your concise final answer]
"""


In [6]:
def run_agent(query, max_steps=10):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Question: {query}"}
    ]
    
    print(f"Starting agent for query: {query}\n")
    
    for step in range(max_steps):
        print(f"--- Step {step + 1} ---")
        
        response = client.chat.completions.create(
            model=model_name,
            messages=messages
        )
        content = response.choices[0].message.content
        messages.append({"role": "assistant", "content": content})
        
        print(f"Agent Thought/Action:\n{content}\n")
        
        if "Final Answer:" in content:
            final_answer = content.split("Final Answer:")[-1].strip()
            print(f"Final Answer: {final_answer}\n")
            return final_answer
        
        # Parse Action from content
        if "Action:" in content:
            try:
                action_part = content.split("Action:")[-1].strip()
                tool_name, tool_input = action_part.split("[", 1)
                tool_name = tool_name.strip()
                tool_input = tool_input.rstrip("]").strip()
                
                print(f"Executing tool: {tool_name} with input: {tool_input}")
                
                if tool_name in tools:
                    result = tools[tool_name](tool_input)
                else:
                    result = "Tool not found"
                
                observation = f"Observation: {result}"
                print(f"{observation}\n")
                messages.append({"role": "user", "content": observation})
            except Exception as e:
                observation = f"Observation: Invalid action format - {str(e)}"
                print(f"{observation}\n")
                messages.append({"role": "user", "content": observation})
        else:
            print("No action taken in this step.\n")
            break  # No action, stop
    
    print("Agent failed to reach a final answer.")
    return "Agent failed to reach a final answer."


In [7]:
# Cell 2: Define the LLM Generator Function
# This function acts as the "answer agent". It takes a list of messages and generates a response using the LLM.
# The messages should include a system prompt like: "You are a helpful assistant that thinks step by step."
# It can also include previous conversations or feedback for improvement.

def llm_generator(messages):
    """
    Generate a response from the LLM based on the input messages.
    
    Args:
    messages (list): List of message dictionaries, e.g., [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}]
    
    Returns:
    str: The generated response content.
    """
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0.1,  # Adjust temperature for creativity

    )
    return response.choices[0].message.content

In [8]:
def reflect_agent(answer):
    """
    Analyze the given answer using the reflective prompt and return the feedback.
    
    Args:
    answer (str): The answer to reflect on.
    
    Returns:
    str: The reflection feedback as a string.
    """
    reflect_prompt = (
        f"คุณเป็นผู้เชี่ยวชาญในการประเมินคำตอบ กรุณาวิเคราะห์คำตอบต่อไปนี้ in thai:\n"
        f"- 'completeness': ประเมินความครบถ้วนของคำตอบ (คะแนน 1-10)\n"
        f"- 'accuracy': ประเมินความถูกต้องของคำตอบ (คะแนน 1-10)\n"
        f"- 'clarity': ประเมินความชัดเจนของคำตอบ (คะแนน 1-10)\n"
        f"- 'strengths': จุดแข็งของคำตอบ\n"
        f"- 'weaknesses': จุดอย่อนของคำตอบ\n"
        f"- 'missing_aspects': สิ่งที่ขาดหายไปในคำตอบ\n"
        f"- 'improvement_suggestions': ข้อเสนอแนะเพื่อปรับปรุงคำตอบ\n\n"
        f"คำตอบ: {answer}\n"
    )
    
    # Messages for the reflect function
    messages = [
        {"role": "system", "content": reflect_prompt}
    ]
    
    # เรียกใช้ LLM เพื่อสร้าง Feedback สำหรับคำตอบ
    feedback = llm_generator(messages)
    return feedback


In [11]:
questions = [
    "ผลคูณของ 147 กับ 258 คือเท่าไหร่?",
    "สรุปประวัติของ AI จากหน้า Wikipedia เป็นภาษาไทย",
    "Elon Musk ก่อตั้งบริษัทใดบ้าง และปัจจุบันมีตำแหน่งอะไรในแต่ละบริษัท?",
    "ค้นหาและสรุปข้อมูลล่าสุดเกี่ยวกับ iPhone รุ่นใหม่",
    "คำนวณค่าของ (15 + 7) * 3 / 2",
    "ข้อมูลเศรษฐกิจของประเทศไทย ณ ปี 2024 เป็นอย่างไรบ้าง? (ให้อ้างอิงแหล่งที่มา)",
    "ดาวเคราะห์ดวงใดในระบบสุริยะที่มีขนาดใหญ่ที่สุดและมีจำนวนดวงจันทร์เท่าใด?",
    "สี่เหลี่ยมจัตุรัสที่มีพื้นที่ 256 ตารางเมตร จะมีความยาวเส้นรอบรูปเท่าใด?",
    "อธิบายแนวคิดหลักของ 'การเรียนรู้ของเครื่อง' (Machine Learning) พร้อมยกตัวอย่าง 2 ตัวอย่าง",
    "เปรียบเทียบข้อดีและข้อเสียของภาษา Python และ Java"
]

for i, q in enumerate(questions):
    print(f"Question {i+1}: {q}")
    answer = run_agent(q)
    print(f"Answer: {answer}\n")
    
    
    feedback = reflect_agent(answer)
    
    print("reflect_agent feedback:")
    print(feedback)
    print("\n\n")
    improvement_prompt = (
        
            f"คุณเป็นผู้เชี่ยวชาญในการปรับปรุงคำตอบที่ให้มา กรุณาปรับปรุงคำตอบต่อไปนี้ตาม feedback ที่ได้รับ:\n\n"
            f"คำถาม: {questions}\n\n"
            f"นี่คือคำตอบก่อนหน้าของคุณ: {answer}\n\n"
            f"นี่คือ feedback จากการประเมิน: {feedback}\n\n"
            f"โปรดปรับปรุงคำตอบของคุณให้ดีขึ้นตาม feedback นี้ โดยคิดทีละขั้นตอน."
        )

    # Messages for the improvement generation
    messages = [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": improvement_prompt}]
    
    improved_answer = llm_generator(messages)
    
    print(f"Improved Answer {i + 1}: {improved_answer}")

Question 1: ผลคูณของ 147 กับ 258 คือเท่าไหร่?
Starting agent for query: ผลคูณของ 147 กับ 258 คือเท่าไหร่?

--- Step 1 ---
Agent Thought/Action:
Thought: I need to multiply the numbers 147 and 258 to find the product.
Action: calculator "147 * 258"

Observation: Invalid action format - not enough values to unpack (expected 2, got 1)

--- Step 2 ---
Agent Thought/Action:
Thought: It seems that I did not provide the correct input. I need to use the calculator tool correctly by formatting it as a single mathematical expression.
Action: calculator "(147 * 258)"

Observation: Invalid action format - not enough values to unpack (expected 2, got 1)

--- Step 3 ---
Agent Thought/Action:
I apologize for the formatting errors in my previous attempts. Let me correct this.

Action: calculator "(147) * (258)"

Observation: Invalid action format - not enough values to unpack (expected 2, got 1)

--- Step 4 ---
Agent Thought/Action:
I realize the previous attempts to calculate the product were incorre